In [1]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9" # replace with your dataset path

In [2]:
# Display training sessions
import glob
import os

sessions = sorted(glob.glob(os.path.join(DATA_DOWNLOAD_DIR, "emg2pose_dataset_mini/*.hdf5")))
sessions

['/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-11_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-11_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-12_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-12_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-13_left.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-13_right.hdf5',
 '/Volumes/Crucial X9/emg2pose_dataset_mini/2022-12-06-16703

In [3]:
# Train model on a subset of sessions (range-based)

import os
import shutil
import subprocess
import pandas as pd
from pathlib import Path

TMP_DIR = DATA_DOWNLOAD_DIR / "emg_sessions"
TMP_DIR.mkdir(exist_ok=True)

# --- parameters ---
start_idx = 0
end_idx = 0  # inclusive

# --- validate + clamp ---
n = len(sessions)
if n == 0:
    raise ValueError("No sessions found")

start_idx = max(0, start_idx)
end_idx = min(n - 1, end_idx)

if start_idx > end_idx:
    raise ValueError("start_idx must be <= end_idx")

selected_sessions = sessions[start_idx:end_idx + 1]
print(f"Training on {len(selected_sessions)} sessions")

# --- clear temp folder ---
for f in TMP_DIR.glob("*"):
    if not f.name.startswith("._"):
        f.unlink()

rows = []

# --- copy sessions + build metadata ---
for session_path in selected_sessions:
    filename = os.path.basename(session_path)
    print("Adding:", filename)

    shutil.copy(session_path, TMP_DIR / filename)

    filename_no_ext = filename.replace(".hdf5", "")
    session_name = filename_no_ext.split("-recording")[0]

    for split in ["train", "val", "test"]:
        rows.append({
            "session": session_name,
            "user": "debug_user",
            "stage": "debug",
            "start": 0,
            "end": 1,
            "side": "right",
            "filename": filename_no_ext,
            "moving_hand": "both",
            "held_out_user": False,
            "held_out_stage": False,
            "split": split,
            "generalization": "debug"
        })

# --- write metadata ---
df = pd.DataFrame(rows)
df.to_csv(TMP_DIR / "metadata.csv", index=False)

print("Created metadata.csv at:", TMP_DIR)



Training on 1 sessions
Adding: 2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_left.hdf5
Created metadata.csv at: /Volumes/Crucial X9/emg_sessions


In [4]:
# --- run training ONCE on full subset ---
subprocess.run([
    "python", "-m", "emg2pose.train",
    "train=True",
    "eval=True",
    "experiment=tracking_vemg2pose",
    "trainer.max_epochs=100",
    "+callbacks.1.dirpath=/Volumes/Crucial X9/emg_checkpoints",
    f"data_location={TMP_DIR}"
])

/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[ext

[2026-04-12 09:52:09,247][__main__][INFO] - 
Config:
data_location: /Volumes/Crucial X9/emg_sessions
seed: 42
batch_size: 64
num_workers: 0
train: true
eval: true
checkpoint: null
monitor_metric: val_loss
monitor_mode: min
loss_weights:
  mae: 1
  fingertip_distance: 0.01
trainer:
  max_epochs: 100
  gradient_clip_val: 1
lr_scheduler: null
callbacks:
- _target_: pytorch_lightning.callbacks.LearningRateMonitor
- _target_: pytorch_lightning.callbacks.ModelCheckpoint
  monitor: ${monitor_metric}
  mode: ${monitor_mode}
  save_last: true
  verbose: true
  dirpath: /Volumes/Crucial X9/emg_checkpoints
- _target_: pytorch_lightning.callbacks.EarlyStopping
  monitor: ${monitor_metric}
  mode: ${monitor_mode}
  patience: 50
  check_on_train_epoch_end: false
  verbose: true
datamodule:
  _target_: emg2pose.lightning.WindowedEmgDataModule
  window_length: 11790
  val_test_window_length: 11790
  padding:
  - 0
  - 0
  skip_ik_failures: true
optimizer:
  _target_: torch.optim.Adam
  lr: 0.001
data_

Missing logger folder: /Volumes/Crucial X9/emg2pose/notebooks/logs/2026-04-12/09-52-09/lightning_logs
/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:653: Checkpoint directory /Volumes/Crucial X9/emg_checkpoints exists and is not empty.

  | Name  | Type            | Params
------------------------------------------
0 | model | StatePoseModule | 6.0 M 
------------------------------------------
6.0 M     Trainable params
0         Non-trainable params
6.0 M     Total params
23.880    Total estimated model params size (MB)
/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]                             

/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:298: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


...

Epoch 99, global step 100: 'val_loss' was not in top 1
`Trainer.fit` stopped: `max_epochs=100` reached.


Validation DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.84it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc          0.00010263722651870921
 val_fingertip_distance     25.589920043945312
        val_jerk          0.00015464324678760022
  val_landmark_distance      15.23615550994873
        val_loss            0.4722394049167633
         val_mae            0.21634024381637573
     val_mae_distal         0.23295609652996063
      val_mae_index          0.218647301197052
       val_mae_mid          0.3112696409225464
     val_mae_middle         0.21465837955474854
      val_mae_pinky         0.28213492035865784
    val_mae_proximal        0.16056762635707855
      val_mae_ring          0.25916942954063416
      val_mae_thumb   

/opt/homebrew/Caskroom/miniconda/base/envs/emg2pose/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.62it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc          0.00010263722651870921
 test_fingertip_distance    25.589920043945312
        test_jerk         0.00015464324678760022
 test_landmark_distance      15.23615550994873
        test_loss           0.4722394049167633
        test_mae            0.21634024381637573
     test_mae_distal        0.23295609652996063
     test_mae_index          0.218647301197052
      test_mae_mid          0.3112696409225464
     test_mae_middle        0.21465837955474854
     test_mae_pinky         0.28213492035865784
    test_mae_proximal       0.16056762635707855
      test_mae_ring         0.25916942954063416
     test_mae_thumb      

CompletedProcess(args=['python', '-m', 'emg2pose.train', 'train=True', 'eval=True', 'experiment=tracking_vemg2pose', 'trainer.max_epochs=100', '+callbacks.1.dirpath=/Volumes/Crucial X9/emg_checkpoints', 'data_location=/Volumes/Crucial X9/emg_sessions'], returncode=0)